# ДЗ 2 — продолжение ассистента из ДЗ 1

Два отдельных файла запуска в корне проекта:
- `assistant.py` — ДЗ 1: погода, курсы ЦБ РФ, Wikipedia, Open Library.
- `assistant_with_memory.py` — ДЗ 1 + ДЗ 2: те же инструменты, LangChain и память фактов.

В JSON сохраняются имя, стек, уровень, цели, предпочтения и город. Переписка
остаётся только в оперативной памяти и логах. Извлечение фактов выполняет LLM
с JSON Schema. Главная LCEL-цепочка: извлечь → сохранить → загрузить профиль → ответить
с использованием инструментов ДЗ 1. Пример с собеседованием не обязателен.

Выберите Python из `.venv`. Зависимости: `python -m pip install -r homework_2/requirements.txt`.
Для API нужен интернет; для сохранения фактов и демонстрации перезапуска сеть не нужна.


In [1]:
import json
import sys
import subprocess
from pathlib import Path
from uuid import uuid4
import ipywidgets as widgets
from IPython.display import display

PROJECT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
               if (path / "assistant_with_memory.py").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import assistant
from assistant_with_memory import AssistantWithMemory, DEFAULT_MEMORY
from homework_2.chat_memory import FileFactMemory, local_models

HOMEWORK = PROJECT / "homework_2"
MODEL_NAME = "qwen2.5-3b"
MODEL_PATH = assistant.MODEL_PATHS[MODEL_NAME]
MEMORY_PATH = DEFAULT_MEMORY
USER_ID = "student"
N_GPU_LAYERS = 0
assert MODEL_PATH.is_file(), f"Нет модели: {MODEL_PATH}"


## Собственная CELL для памяти и диагностики

Загрузка профиля, изменения фактов, ошибки и служебный вывод библиотек направляются
в отдельный блок ниже. В основном чате будут только реплики пользователя и ассистента.


In [2]:
service_cell = widgets.Output(layout={"border": "1px solid #888", "padding": "8px"})
service_lines = []
display(widgets.VBox([widgets.HTML("<b>CELL: память и диагностика</b>"), service_cell]))

def write_service(text):
    service_lines.append(text)
    service_cell.outputs = ({"name": "stdout", "output_type": "stream",
                             "text": "".join(service_lines)},)

def show_event(event):
    write_service(json.dumps(event, ensure_ascii=False, indent=2) + "\n")


[
  {
    "event": "restart_verified",
    "separate_processes": true,
    "user_id": "demo_agent_c05e4ade",
    "facts": {
      "name": "Анна",
      "stack": "Python",
      "city": "Казань"
    }
  },
  {
    "event": "demo_memory_snapshot",
    "facts": {
      "name": "Анна",
      "stack": "Python",
      "city": "Казань"
    }
  }
]


## Демонстрация перезапуска в двух процессах

Первый процесс запускает **ассистента ДЗ 1 + ДЗ 2**, запоминает факты вымышленной
Анны и завершается. Второй получает только файл памяти, без истории сообщений,
и отвечает, что помнит. Для каждого выполнения создаётся отдельный демонстрационный
профиль; рабочий профиль `student` не меняется. Демонстрационный файл `memory.json`
в папке notebook входит в материалы для сдачи.


In [3]:
DEMO_MEMORY = HOMEWORK / "memory.json"
DEMO_USER = "demo_agent_" + uuid4().hex[:8]

def run_process(message):
    result = subprocess.run([
        sys.executable, "-X", "utf8", str(PROJECT / "assistant_with_memory.py"),
        "--model", MODEL_NAME, "--memory", str(DEMO_MEMORY), "--user", DEMO_USER,
        "--gpu-layers", str(N_GPU_LAYERS), "--message", message,
    ], cwd=PROJECT, capture_output=True, text=True, encoding="utf-8", timeout=240)
    write_service(result.stderr)
    startup, separator, answer = result.stdout.partition("Ассистент: ")
    write_service(startup)
    if result.returncode or not separator:
        raise RuntimeError("Ошибка демонстрации; подробности в CELL диагностики")
    return answer.strip()

first_message = "Меня зовут Анна. Я живу в Казани. Я пишу на Python."
print("ПЕРВЫЙ ПРОЦЕСС\nВы:", first_message)
print("Ассистент:", run_process(first_message))
saved = FileFactMemory(DEMO_MEMORY, DEMO_USER).load()
assert saved.name == "Анна" and "Python" in (saved.stack or "") and saved.city

print("\nПЕРВЫЙ ПРОЦЕСС ЗАВЕРШЁН. ЗАПУЩЕН ВТОРОЙ.")
question = "Как меня зовут, в каком городе я живу и на чём пишу?"
print("Вы:", question)
second_answer = run_process(question)
print("Ассистент:", second_answer)
assert "Анна" in second_answer and "Python" in second_answer
assert FileFactMemory(DEMO_MEMORY, DEMO_USER).load() == saved
show_event({"event": "restart_verified", "separate_processes": True,
            "user_id": DEMO_USER, "facts": saved.model_dump(exclude_none=True)})


ПЕРВЫЙ ПРОЦЕСС
Вы: Меня зовут Анна. Я живу в Казани. Я пишу на Python.


Ассистент: Понял, Анна, ты живешь в Казани и пишешь на Python.

ПЕРВЫЙ ПРОЦЕСС ЗАВЕРШЁН. ЗАПУЩЕН ВТОРОЙ.
Вы: Как меня зовут, в каком городе я живу и на чём пишу?


Ассистент: Ваше имя — Анна, вы живете в Казани, и вы пишете на Python.


## Модель и LangChain-цепочка

Извлечение фактов и обычные ответы используют одну локальную GGUF-модель.
Класс `AssistantWithMemory` и CLI находятся во втором файле запуска.
Он повторно использует функции и разбор `$function$…$function$` из `assistant.py`.
Рабочая память — `agent_memory.json` в корне проекта; она отделена от демонстрационной.


In [4]:
with service_cell:
    if "local_model" in globals():
        local_model.close()
    if "chat_logger" in globals():
        for handler in chat_logger.handlers[:]:
            handler.close()
            chat_logger.removeHandler(handler)
    chat_logger, chat_log_path = assistant.setup_logger(MODEL_NAME + "_with_memory_notebook")
    _, json_llm, local_model = local_models(MODEL_PATH, N_GPU_LAYERS, n_ctx=8192)
    chat = AssistantWithMemory(local_model, json_llm, FileFactMemory(MEMORY_PATH, USER_ID),
                               logger=chat_logger, on_event=show_event)


## Основной чат: возможности ДЗ 1 + память

Попробуйте: «Меня зовут Анна. Я живу в Казани», затем «Какая погода в моём городе?».
Также доступны курсы валют, поиск статей и книг. После перезапуска спросите
«Как меня зовут и где я живу?». Неизвестные факты агент не должен придумывать.

`/clear` очищает короткую историю, сохраняя факты. `/memory` показывает профиль
в служебной CELL. «Пока» завершает чат и освобождает модель. Предложения с `?`
не используются для записи новых фактов: отделяйте факт от вопроса точкой.


In [5]:
chat_cell = widgets.Output(layout={"border": "1px solid #bbb", "padding": "8px"})
message_input = widgets.Textarea(placeholder="Ваше сообщение", layout={"width": "90%"})
send_button = widgets.Button(description="Отправить", button_style="primary")
status = widgets.Label()
display(widgets.VBox([chat_cell, message_input, send_button, status]))
chat_cell.append_stdout("Ассистент: Привет! Могу найти погоду, курс, статью или книгу и запомнить сведения о вас.\n\n")

def send_message(_):
    text = message_input.value.strip()
    if not text:
        return
    message_input.value = ""
    chat_cell.append_stdout("Вы: " + text + "\n")
    if text.casefold() in {"exit", "quit", "выход", "конец"} or assistant.social_reply(text) == "До свидания!":
        chat_cell.append_stdout("Ассистент: До свидания!\n")
        message_input.disabled = send_button.disabled = True
        local_model.close()
        for handler in chat_logger.handlers[:]:
            handler.close()
            chat_logger.removeHandler(handler)
        return
    if text == "/clear":
        chat.clear_dialogue()
        chat_cell.append_stdout("Новый диалог. Факты сохранены.\n")
        return
    if text == "/memory":
        show_event({"event": "memory_snapshot", "facts": chat.memory.load().model_dump(exclude_none=True)})
        return
    send_button.disabled = True
    status.value = "Подготовка ответа…"
    try:
        with service_cell:
            answer = chat.chain.invoke(text)
        chat_cell.append_stdout("Ассистент: " + answer + "\n\n")
    except Exception as exc:
        show_event({"event": "error", "message": str(exc)})
        status.value = "Ошибка: подробности в CELL диагностики."
    else:
        status.value = ""
    finally:
        send_button.disabled = False

send_button.on_click(send_message)


## Сохранённые факты и материалы для сдачи

В памяти нет сообщений чата и результатов API. Значения `null` не стирают известные
факты, исправления заменяют значения, а `user_id` разделяет профили. Структура JSON
проверяется Pydantic; точность извлечения смысла зависит от небольшой локальной модели.

Сдать этот notebook с результатами, демонстрационный `memory.json`, оба файла запуска
`assistant.py` и `assistant_with_memory.py`, а также общие модули `public_api_tools.py`
и `homework_2/chat_memory.py`. Первый файл запуска работает самостоятельно без
LangChain и файловой памяти, второй добавляет требования ДЗ 2.


In [6]:
show_event({"event": "demo_memory_snapshot", "path": str(DEMO_MEMORY),
            "facts": FileFactMemory(DEMO_MEMORY, DEMO_USER).load().model_dump(exclude_none=True)})
